# NB2 — Per-Subject Fine-Tuning & Evaluation

**Goal:** Take the population prior from NB1 and adapt it to each individual subject using
a small calibration set (runs 1–2). Evaluate on run 3 (held out from both pre-training and fine-tuning).

**Three-way comparison:**
1. **EEGNet fine-tuned** — NB1 checkpoint → 30-epoch fine-tune on calibration data
2. **Riemannian MDM** — `Covariances(lwf)` + `MDM(riemann)` fit on same calibration data (classical baseline)
3. **EEGNet zero-shot** — NB1 checkpoint applied directly, no fine-tuning (sanity check)

**Calibration budget curve:** repeat fine-tune at N ∈ {5, 10, 20, 30, 45, 60} trials to show
how accuracy scales with labelled data — the clinically relevant user-burden question.

**Preprocessing alignment:**
- Same bandpass (1–79 Hz), same epoch window (1–3 s), same subject exclusions as NB1
- EA re-derived from calibration runs only — no leakage from test run
- MDM receives identical preprocessing (1–79 Hz + EA) for architecture-controlled comparison

**Expected runtime:** ~30–45 min (fine-tune is 30 epochs vs 150 in NB1)

In [1]:
import os, gc, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.stats import wilcoxon

import mne
mne.set_log_level('WARNING')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from braindecode.models import EEGNet

from pyriemann.estimation import Covariances
from pyriemann.classification import MDM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

os.makedirs('results', exist_ok=True)

PyTorch 2.5.1+cu121  |  device: cuda
  GPU: NVIDIA GeForce GTX 1080 Ti


In [2]:
# ── Data (must match NB1 exactly) ─────────────────────────────────────────────
TASK2_RUNS  = [4, 8, 12]
CAL_RUNS    = [4, 8]      # calibration: runs 1–2
TEST_RUNS   = [12]        # test: run 3 — never seen during pre-training or fine-tuning
SFREQ       = 160
TMIN, TMAX  = 1.0, 3.0
N_TIMES     = int((TMAX - TMIN) * SFREQ) + 1
N_CHANS     = 64
BANDHI      = 79.0
KNOWN_BAD   = {88, 89, 92, 100, 104, 106}

# ── EEGNet (must match NB1 exactly) ───────────────────────────────────────────
F1, D, F2         = 8, 2, 16
KERNEL_LENGTH     = 64
DROP_PROB         = 0.25

# ── Fine-tuning ───────────────────────────────────────────────────────────────
FT_EPOCHS   = 30
FT_LR       = 5e-4       # lower than NB1 — we're adapting, not training from scratch
FT_WDECAY   = 1e-4
BATCH_SIZE  = 16         # smaller — calibration sets are small

# ── Calibration budget sweep ──────────────────────────────────────────────────
# Cal runs (runs 1–2) yield ~14 trials/class per subject — cap budgets accordingly
BUDGETS     = [3, 5, 7, 10, 12]

print(f'Cal runs: {CAL_RUNS}  |  Test run: {TEST_RUNS}')
print(f'Fine-tune: {FT_EPOCHS} epochs  lr={FT_LR}  batch={BATCH_SIZE}')
print(f'Budget sweep: {BUDGETS} trials/class')

Cal runs: [4, 8]  |  Test run: [12]
Fine-tune: 30 epochs  lr=0.0005  batch=16
Budget sweep: [3, 5, 7, 10, 12] trials/class


In [3]:
def load_subject_split(sub):
    """
    Load one subject, return calibration and test splits separately.
    EA is derived from calibration data only — no leakage from test run.
    Returns: (X_cal, y_cal, X_te, y_te) all in µV, or (None,)*4 on failure.
    """
    try:
        def _load_runs(runs):
            paths = mne.datasets.eegbci.load_data(sub, runs=runs, verbose=False)
            raws  = [mne.io.read_raw_edf(p, preload=True, verbose=False) for p in paths]
            raw   = mne.concatenate_raws(raws)
            mne.datasets.eegbci.standardize(raw)
            raw.filter(1., BANDHI, verbose=False)
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            motor_ids = {k: v for k, v in event_id.items() if k in ('T1', 'T2')}
            if not motor_ids:
                return None, None
            epochs = mne.Epochs(raw, events, event_id=motor_ids,
                                tmin=TMIN, tmax=TMAX, baseline=None,
                                preload=True, verbose=False)
            X = epochs.get_data(units='uV').astype(np.float32)
            y = (epochs.events[:, 2] == motor_ids.get('T2', -1)).astype(np.int64)
            return X, y

        X_cal, y_cal = _load_runs(CAL_RUNS)
        X_te,  y_te  = _load_runs(TEST_RUNS)
        if X_cal is None or X_te is None:
            return None, None, None, None
        return X_cal, y_cal, X_te, y_te
    except Exception:
        return None, None, None, None


def ea_invsqrt(X):
    n_ep, n_ch, n_t = X.shape
    R = np.mean(
        [X[i].astype(np.float64) @ X[i].astype(np.float64).T / n_t
         for i in range(n_ep)], axis=0
    )
    w, v = eigh(R)
    w    = np.maximum(w, 1e-10)
    return ((v * (w ** -0.5)) @ v.T).astype(np.float32)


def apply_ea(X, R_invsqrt):
    return np.einsum('ij,njt->nit',
                     R_invsqrt.astype(np.float64),
                     X.astype(np.float64)).astype(np.float32)


# Load NB1 subject list from saved results
nb1_results = dict(np.load('results/loso_results.npy', allow_pickle=True).item())
SUBS        = sorted(nb1_results.keys())
print(f'Subjects from NB1: {len(SUBS)}')
print('Loading per-subject calibration / test splits …')

t0       = time.time()
splits   = {}   # sub -> (X_cal_ea, y_cal, X_te_ea, y_te)
failed   = []

for sub in SUBS:
    X_cal, y_cal, X_te, y_te = load_subject_split(sub)
    if X_cal is None or len(y_cal) < 6 or len(y_te) < 2:
        failed.append(sub)
        continue
    # EA from calibration runs only
    R          = ea_invsqrt(X_cal)
    X_cal_ea   = apply_ea(X_cal, R)
    X_te_ea    = apply_ea(X_te,  R)
    splits[sub] = (X_cal_ea, y_cal, X_te_ea, y_te)

SUBS = [s for s in SUBS if s in splits]
print(f'Ready: {len(SUBS)} subjects in {time.time()-t0:.0f}s')
if failed:
    print(f'Failed: {failed}')

Subjects from NB1: 102
Loading per-subject calibration / test splits …


Ready: 102 subjects in 114s


In [4]:
def make_eegnet():
    return EEGNet(
        n_chans=N_CHANS, n_outputs=2, n_times=N_TIMES,
        final_conv_length='auto',
        F1=F1, D=D, F2=F2,
        kernel_length=KERNEL_LENGTH,
        drop_prob=DROP_PROB,
    ).to(DEVICE)


def load_checkpoint(sub):
    ckpt = f'checkpoints/fold_{sub:03d}.pt'
    model = make_eegnet()
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model


@torch.no_grad()
def eval_acc(model, X, y):
    model.eval()
    preds = model(torch.FloatTensor(X).to(DEVICE)).argmax(1).cpu().numpy()
    return float((preds == y).mean() * 100.)


def finetune(model, X_cal, y_cal, n_trials=None):
    """
    Fine-tune model on calibration data. If n_trials is set, subsample
    that many trials per class (for calibration budget curve).
    Returns fine-tuned model (modifies in-place and returns).
    """
    if n_trials is not None:
        idx = []
        for cls in [0, 1]:
            cls_idx = np.where(y_cal == cls)[0]
            n       = min(n_trials, len(cls_idx))
            idx.extend(np.random.choice(cls_idx, n, replace=False))
        idx    = np.array(idx)
        X_cal  = X_cal[idx]
        y_cal  = y_cal[idx]

    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_cal), torch.LongTensor(y_cal)),
        batch_size=min(BATCH_SIZE, len(y_cal)),
        shuffle=True, drop_last=False,
        pin_memory=(DEVICE == 'cuda'),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR, weight_decay=FT_WDECAY)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for _ in range(FT_EPOCHS):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            optimizer.step()
    return model


print('Model utilities ready.')

Model utilities ready.


In [5]:
# ── Main evaluation loop ───────────────────────────────────────────────────────
# For each subject: zero-shot, full fine-tune, MDM baseline

np.random.seed(42)

results = {}   # sub -> {'zeroshot': float, 'finetune': float, 'mdm': float}

# Resume support
RESULTS_PATH = 'results/nb2_main_results.npy'
if os.path.exists(RESULTS_PATH):
    results = dict(np.load(RESULTS_PATH, allow_pickle=True).item())
    print(f'Resuming — {len(results)} subjects already done')

hdr = f'{"Sub":>4} | {"ZeroShot":>8} | {"FineTune":>8} | {"MDM":>6} | {"FT gain":>7}'
print(hdr)
print('-' * len(hdr))

t_total = time.time()

for sub in SUBS:
    if sub in results:
        r = results[sub]
        print(f'{sub:>4} | (skipped — zs={r["zeroshot"]:.1f}% ft={r["finetune"]:.1f}% mdm={r["mdm"]:.1f}%)')
        continue

    X_cal, y_cal, X_te, y_te = splits[sub]

    # ── Zero-shot: NB1 checkpoint, no fine-tuning ──────────────────────────
    model     = load_checkpoint(sub)
    acc_zs    = eval_acc(model, X_te, y_te)

    # ── Fine-tune: all calibration data ───────────────────────────────────
    finetune(model, X_cal, y_cal)
    acc_ft    = eval_acc(model, X_te, y_te)
    del model; gc.collect(); torch.cuda.empty_cache()

    # ── MDM baseline ──────────────────────────────────────────────────────
    cov_est   = Covariances(estimator='lwf')
    C_cal     = cov_est.fit_transform(X_cal)
    C_te      = cov_est.transform(X_te)
    mdm       = MDM(metric='riemann', n_jobs=1).fit(C_cal, y_cal)
    acc_mdm   = float(mdm.score(C_te, y_te) * 100.)

    results[sub] = {'zeroshot': acc_zs, 'finetune': acc_ft, 'mdm': acc_mdm}
    np.save(RESULTS_PATH, results)

    print(f'{sub:>4} | {acc_zs:>8.1f} | {acc_ft:>8.1f} | {acc_mdm:>6.1f} | {acc_ft-acc_zs:>+6.1f}pp')

print('-' * len(hdr))
zs  = np.array([results[s]['zeroshot'] for s in SUBS])
ft  = np.array([results[s]['finetune'] for s in SUBS])
mdm = np.array([results[s]['mdm']      for s in SUBS])

for label, arr in [('Zero-shot', zs), ('Fine-tune', ft), ('MDM', mdm)]:
    print(f'{label:>10}: {arr.mean():.2f} ± {arr.std():.2f}%  median {np.median(arr):.2f}%')

stat_ft_mdm, p_ft_mdm = wilcoxon(ft, mdm)
stat_ft_zs,  p_ft_zs  = wilcoxon(ft, zs)
print(f'\nWilcoxon fine-tune vs MDM      : p = {p_ft_mdm:.4f}')
print(f'Wilcoxon fine-tune vs zero-shot: p = {p_ft_zs:.4f}')
print(f'Total time: {(time.time()-t_total)/60:.1f} min')

Resuming — 102 subjects already done
 Sub | ZeroShot | FineTune |    MDM | FT gain
---------------------------------------------
   1 | (skipped — zs=80.0% ft=93.3% mdm=53.3%)
   2 | (skipped — zs=80.0% ft=80.0% mdm=80.0%)
   3 | (skipped — zs=60.0% ft=60.0% mdm=46.7%)
   4 | (skipped — zs=86.7% ft=93.3% mdm=46.7%)
   5 | (skipped — zs=60.0% ft=60.0% mdm=66.7%)
   6 | (skipped — zs=66.7% ft=73.3% mdm=53.3%)
   7 | (skipped — zs=93.3% ft=93.3% mdm=93.3%)
   8 | (skipped — zs=80.0% ft=66.7% mdm=46.7%)
   9 | (skipped — zs=80.0% ft=73.3% mdm=66.7%)
  10 | (skipped — zs=80.0% ft=73.3% mdm=53.3%)
  11 | (skipped — zs=66.7% ft=60.0% mdm=46.7%)
  12 | (skipped — zs=100.0% ft=100.0% mdm=73.3%)
  13 | (skipped — zs=80.0% ft=73.3% mdm=46.7%)
  14 | (skipped — zs=66.7% ft=66.7% mdm=46.7%)
  15 | (skipped — zs=93.3% ft=100.0% mdm=73.3%)
  16 | (skipped — zs=66.7% ft=60.0% mdm=66.7%)
  17 | (skipped — zs=46.7% ft=53.3% mdm=53.3%)
  18 | (skipped — zs=86.7% ft=80.0% mdm=33.3%)
  19 | (skipped — zs=7

In [6]:
# ── Calibration budget curve ───────────────────────────────────────────────────
# For each budget N (trials per class): fine-tune from NB1 checkpoint on N trials,
# evaluate on test run. Average over 5 random subsamples per subject per budget.

N_SEEDS      = 5
BUDGET_PATH  = 'results/nb2_budget_results.npy'

if os.path.exists(BUDGET_PATH):
    budget_results = dict(np.load(BUDGET_PATH, allow_pickle=True).item())
    print(f'Budget results loaded — budgets done: {sorted(budget_results.keys())}')
else:
    budget_results = {}   # budget -> list of per-subject mean accs

print('\nCalibration budget sweep …')

for n_trials in BUDGETS:
    if n_trials in budget_results:
        print(f'  N={n_trials:>3}: {np.mean(budget_results[n_trials]):.1f}% (cached)')
        continue

    sub_accs = []
    for sub in SUBS:
        X_cal, y_cal, X_te, y_te = splits[sub]
        # Skip if not enough trials for this budget
        min_cls = min((y_cal == 0).sum(), (y_cal == 1).sum())
        if min_cls < n_trials:
            continue

        seed_accs = []
        for seed in range(N_SEEDS):
            np.random.seed(seed * 100 + sub)
            model = load_checkpoint(sub)
            finetune(model, X_cal, y_cal, n_trials=n_trials)
            seed_accs.append(eval_acc(model, X_te, y_te))
            del model; gc.collect(); torch.cuda.empty_cache()

        sub_accs.append(np.mean(seed_accs))

    budget_results[n_trials] = sub_accs
    np.save(BUDGET_PATH, budget_results)
    print(f'  N={n_trials:>3} trials/class → {np.mean(sub_accs):.2f} ± {np.std(sub_accs):.2f}%')

# MDM budget curve — same subsamples
MDM_BUDGET_PATH = 'results/nb2_mdm_budget_results.npy'
if os.path.exists(MDM_BUDGET_PATH):
    mdm_budget_results = dict(np.load(MDM_BUDGET_PATH, allow_pickle=True).item())
    print(f'MDM budget results loaded — budgets done: {sorted(mdm_budget_results.keys())}')
else:
    mdm_budget_results = {}

print('\nMDM calibration budget sweep …')

for n_trials in BUDGETS:
    if n_trials in mdm_budget_results:
        print(f'  N={n_trials:>3}: {np.mean(mdm_budget_results[n_trials]):.1f}% (cached)')
        continue

    sub_accs = []
    for sub in SUBS:
        X_cal, y_cal, X_te, y_te = splits[sub]
        min_cls = min((y_cal == 0).sum(), (y_cal == 1).sum())
        if min_cls < n_trials:
            continue

        seed_accs = []
        for seed in range(N_SEEDS):
            np.random.seed(seed * 100 + sub)
            idx = []
            for cls in [0, 1]:
                cls_idx = np.where(y_cal == cls)[0]
                idx.extend(np.random.choice(cls_idx, n_trials, replace=False))
            idx  = np.array(idx)
            Xs   = X_cal[idx];  ys = y_cal[idx]
            cov_est = Covariances(estimator='lwf')
            C_tr    = cov_est.fit_transform(Xs)
            C_te    = cov_est.transform(X_te)
            acc     = MDM(metric='riemann', n_jobs=1).fit(C_tr, ys).score(C_te, y_te) * 100.
            seed_accs.append(float(acc))

        sub_accs.append(np.mean(seed_accs))

    mdm_budget_results[n_trials] = sub_accs
    np.save(MDM_BUDGET_PATH, mdm_budget_results)
    print(f'  MDM N={n_trials:>3} trials/class → {np.mean(sub_accs):.2f} ± {np.std(sub_accs):.2f}%')


Calibration budget sweep …


  N=  3 trials/class → 73.07 ± 15.07%


  N=  5 trials/class → 73.56 ± 14.88%


  N=  7 trials/class → 73.53 ± 15.51%


  N= 10 trials/class → 73.76 ± 15.29%


...

  MDM N=  3 trials/class → 53.25 ± 5.58%


  MDM N=  5 trials/class → 53.45 ± 6.52%


  MDM N=  7 trials/class → 54.80 ± 7.81%


  MDM N= 10 trials/class → 56.31 ± 9.17%


  MDM N= 12 trials/class → 56.88 ± 10.74%


In [7]:
# ── Figures ───────────────────────────────────────────────────────────────────
results        = dict(np.load('results/nb2_main_results.npy',        allow_pickle=True).item())
budget_results = dict(np.load('results/nb2_budget_results.npy',      allow_pickle=True).item())
mdm_budget     = dict(np.load('results/nb2_mdm_budget_results.npy',  allow_pickle=True).item())
nb1_results    = dict(np.load('results/loso_results.npy',            allow_pickle=True).item())

SUBS_used = sorted(results.keys())
zs  = np.array([results[s]['zeroshot'] for s in SUBS_used])
ft  = np.array([results[s]['finetune'] for s in SUBS_used])
mdm = np.array([results[s]['mdm']      for s in SUBS_used])

INK   = '#1A1A1A'
OX    = '#002147'
SEPIA = '#704214'
TEAL  = '#008080'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('white')

# ── Panel 1: Per-subject scatter — zero-shot vs fine-tuned ───────────────────
ax = axes[0]
ax.scatter(zs, ft, color=OX, alpha=0.6, s=20, zorder=3)
lim = (30, 100)
ax.plot(lim, lim, '--', color=INK, lw=1, alpha=0.4)
ax.set_xlim(*lim); ax.set_ylim(*lim)
ax.set_xlabel('Zero-shot accuracy (%)',   color=INK)
ax.set_ylabel('Fine-tuned accuracy (%)',  color=INK)
ax.set_title(f'Zero-shot vs Fine-tuned\n'
             f'Mean gain: {(ft-zs).mean():+.1f} pp  (p={wilcoxon(ft,zs)[1]:.4f})',
             color=INK, fontsize=10)
ax.set_facecolor('white')

# ── Panel 2: Method comparison boxplot ───────────────────────────────────────
ax = axes[1]
bp = ax.boxplot([zs, ft, mdm],
                labels=['Zero-shot\n(NB1)', 'Fine-tuned\n(NB2)', 'MDM\n(Riemannian)'],
                patch_artist=True, widths=0.5,
                medianprops=dict(color=SEPIA, lw=2))
colors = [TEAL, OX, '#888888']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.axhline(50, color=INK, lw=1, ls=':', alpha=0.4)
ax.set_ylabel('Test accuracy (%)', color=INK)
ax.set_title(f'Method comparison  N={len(SUBS_used)} subjects\n'
             f'FT vs MDM: p={wilcoxon(ft,mdm)[1]:.4f}',
             color=INK, fontsize=10)
ax.set_facecolor('white')

# ── Panel 3: Calibration budget curve ────────────────────────────────────────
ax = axes[2]
budgets = sorted(budget_results.keys())

ft_means  = [np.mean(budget_results[n]) for n in budgets]
ft_sems   = [np.std(budget_results[n]) / np.sqrt(len(budget_results[n])) for n in budgets]
mdm_means = [np.mean(mdm_budget[n])    for n in budgets]
mdm_sems  = [np.std(mdm_budget[n])     / np.sqrt(len(mdm_budget[n]))    for n in budgets]

ax.errorbar(budgets, ft_means,  yerr=ft_sems,  color=OX,        lw=2, marker='o', ms=5, label='EEGNet fine-tuned', capsize=3)
ax.errorbar(budgets, mdm_means, yerr=mdm_sems, color='#888888', lw=2, marker='s', ms=5, label='MDM (Riemannian)',   capsize=3, ls='--')
ax.axhline(zs.mean(),  color=TEAL,  lw=1.5, ls='--', alpha=0.7, label=f'Zero-shot mean ({zs.mean():.1f}%)')
ax.axhline(50,         color=INK,   lw=1,   ls=':',  alpha=0.4, label='Chance')
ax.set_xlabel('Calibration trials per class', color=INK)
ax.set_ylabel('Test accuracy (%)',            color=INK)
ax.set_title('Calibration budget curve\n(mean ± SEM across subjects)', color=INK, fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.set_facecolor('white')

plt.tight_layout()
plt.savefig('results/fig_nb2_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/fig_nb2_evaluation.png')

Saved → results/fig_nb2_evaluation.png


## Summary

| Method | Mean ± SD | vs NB1 zero-shot | vs MDM |
|---|---|---|---|
| NB1 zero-shot | — | baseline | — |
| MDM (Riemannian, lwf) | — | — | — |
| **EEGNet fine-tuned** | — | **see above** | **see above** |

*(Fill in after running)*

**Note on MDM:** MDM receives the same 1–79 Hz broadband + EA preprocessing as EEGNet for
an architecture-controlled comparison. Its optimal preprocessing would be mu/beta bandpass
(8–30 Hz) without EA; this likely underestimates peak Riemannian performance by ~3–5 pp.

**What changes going into NB3:**
- Replace NB1 standard pre-training with Reptile meta-training
- NB3 checkpoint drops into this notebook in place of NB1 checkpoint → second line on budget curve